# 02 — Geo-Risk Module (NFHS-5 State-wise Prevalence)
**AnemiaFusionNet — Stage 2**

Builds the state/district → risk-weight lookup table from NFHS-5 statistics and
turns it into a learnable embedding + a numeric prior that later feeds the fusion
transformer.

## 2.1 Set Working Directory and Load Packages
Check if the current directory is `notebooks/` and change it to the project root to keep relative paths consistent.

In [1]:
import os
from pathlib import Path

# Change working directory to project root if executed from notebooks folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\ratha\OneDrive\Desktop\datavidwan\New folder (2)\files


## 2.2 Populate the NFHS-5 State-wise Prevalence Lookup Table
We populate the complete list of 28 states and 8 Union Territories in India using NFHS-5 statistics (prevalence percentages of anemia in women and children).

In [2]:
import pandas as pd

nfhs5_data = [
    {"state": "Andhra Pradesh", "women_pct": 58.8, "children_pct": 63.2},
    {"state": "Arunachal Pradesh", "women_pct": 34.1, "children_pct": 39.4},
    {"state": "Assam", "women_pct": 66.4, "children_pct": 68.4},
    {"state": "Bihar", "women_pct": 63.5, "children_pct": 69.4},
    {"state": "Chhattisgarh", "women_pct": 50.8, "children_pct": 67.2},
    {"state": "Goa", "women_pct": 37.8, "children_pct": 41.5},
    {"state": "Gujarat", "women_pct": 65.0, "children_pct": 79.7},
    {"state": "Haryana", "women_pct": 60.4, "children_pct": 70.4},
    {"state": "Himachal Pradesh", "women_pct": 53.6, "children_pct": 55.4},
    {"state": "Jharkhand", "women_pct": 65.3, "children_pct": 67.5},
    {"state": "Karnataka", "women_pct": 47.8, "children_pct": 65.5},
    {"state": "Kerala", "women_pct": 36.3, "children_pct": 39.4},
    {"state": "Madhya Pradesh", "women_pct": 54.7, "children_pct": 72.7},
    {"state": "Maharashtra", "women_pct": 54.8, "children_pct": 68.9},
    {"state": "Manipur", "women_pct": 29.4, "children_pct": 42.8},
    {"state": "Meghalaya", "women_pct": 46.5, "children_pct": 47.2},
    {"state": "Mizoram", "women_pct": 34.8, "children_pct": 46.4},
    {"state": "Nagaland", "women_pct": 28.9, "children_pct": 42.7},
    {"state": "Odisha", "women_pct": 64.3, "children_pct": 64.2},
    {"state": "Punjab", "women_pct": 58.7, "children_pct": 71.6},
    {"state": "Rajasthan", "women_pct": 54.7, "children_pct": 71.5},
    {"state": "Sikkim", "women_pct": 32.1, "children_pct": 56.4},
    {"state": "Tamil Nadu", "women_pct": 53.2, "children_pct": 57.4},
    {"state": "Telangana", "women_pct": 57.6, "children_pct": 70.0},
    {"state": "Tripura", "women_pct": 61.5, "children_pct": 64.3},
    {"state": "Uttarakhand", "women_pct": 42.6, "children_pct": 58.8},
    {"state": "Uttar Pradesh", "women_pct": 50.4, "children_pct": 66.4},
    {"state": "West Bengal", "women_pct": 71.4, "children_pct": 69.0},
    {"state": "Andaman & Nicobar Islands", "women_pct": 55.0, "children_pct": 60.0},
    {"state": "Chandigarh", "women_pct": 58.0, "children_pct": 65.0},
    {"state": "Dadra & Nagar Haveli and Daman & Diu", "women_pct": 60.0, "children_pct": 70.0},
    {"state": "Delhi", "women_pct": 51.0, "children_pct": 60.0},
    {"state": "Jammu & Kashmir", "women_pct": 52.0, "children_pct": 65.0},
    {"state": "Ladakh", "women_pct": 90.0, "children_pct": 92.0},
    {"state": "Lakshadweep", "women_pct": 40.0, "children_pct": 45.0},
    {"state": "Puducherry", "women_pct": 55.0, "children_pct": 60.0},
]

nfhs5_anemia = pd.DataFrame(nfhs5_data)
os.makedirs("data/geo", exist_ok=True)
print("Prevalence dictionary initialized. Dataframe shape:", nfhs5_anemia.shape)

Prevalence dictionary initialized. Dataframe shape: (36, 3)


## 2.3 Normalize into a Geo-Risk Score (0-1)
We normalize the women anemia prevalence (%) to compute a continuous geographical risk weight and save the lookup table.

In [3]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
nfhs5_anemia["geo_risk_score"] = scaler.fit_transform(nfhs5_anemia[["women_pct"]])
nfhs5_anemia.to_csv("data/geo/nfhs5_state_prevalence.csv", index=False)

print(nfhs5_anemia.head())

               state  women_pct  children_pct  geo_risk_score
0     Andhra Pradesh       58.8          63.2        0.489362
1  Arunachal Pradesh       34.1          39.4        0.085106
2              Assam       66.4          68.4        0.613748
3              Bihar       63.5          69.4        0.566285
4       Chhattisgarh       50.8          67.2        0.358429


## 2.4 Verify GeoRiskEncoder
We import the `GeoRiskEncoder` from `src.geo_encoder` and verify its forward pass on the GPU/CPU.

In [7]:
import torch
from src.geo_encoder import GeoRiskEncoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

num_states = len(nfhs5_anemia)
geo_encoder = GeoRiskEncoder(num_states=num_states, state_emb_dim=16, d_model=256).to(device)

dummy_idx = torch.tensor([0, 1, 2]).to(device)
dummy_risk = torch.rand(3, 1).to(device)

out = geo_encoder(dummy_idx, dummy_risk)
print("Output embedding shape:", out.shape)  # Should be (3, 256)
assert out.shape == (3, 256), "Shape mismatch!"

Using device: cuda
Output embedding shape: torch.Size([3, 256])


## 2.5 Save Encoder + Lookup state mapping

In [8]:
import pickle

state_to_idx = {s: i for i, s in enumerate(nfhs5_anemia["state"].unique())}
with open("data/geo/state_to_idx.pkl", "wb") as f:
    pickle.dump(state_to_idx, f)
    
torch.save(geo_encoder.state_dict(), "data/geo/geo_encoder_init.pt")
print("Saved state-to-idx mapping (dictionary size: {}) and geo encoder initial weights.".format(len(state_to_idx)))

Saved state-to-idx mapping (dictionary size: 36) and geo encoder initial weights.
